***Small Language Models -- Introduction***

*Based on a notebook originally written by Pratheerth Padman*

In [1]:
# Load Hugging Face Token from shared environment
# 
from dotenv import load_dotenv
import os
load_dotenv('../shared/.env')
key = os.getenv('HF_TOKEN')
os.environ["HF_TOKEN"] = key

In [2]:
# install required libraries  
# This is not on the image so we need to install


!pip install accelerate>=0.26.0 --quiet
print("Libraries installed!")

Libraries installed!


In [3]:
# Now we import the packages needed.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check to make certain we have a GPU
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))



CUDA available: True
Device count: 1
Device name: Tesla T4


In [4]:
import time

In [5]:
#def get_gpu_mem_mb():
    # Returns currently allocated GPU memory in MB
#    return torch.cuda.memory_allocated() / (1024 ** 2)

The Models have been downloaded and stored in a shared directory
* Llama-3.2-1B-Instruct
* gemma-2-2b-it
* Qwen/Qwen2.5-0.5B-Instruct

There are also quantized gguf versions of each.


In [ ]:
#base_mem = get_gpu_mem_mb()

In [ ]:
#print(f"Base Memory: {base_mem:.2f} MB")

In [8]:
local_model_path = "/home/jovyan/shared/models/meta-llama--Llama-3.2-1B-Instruct"

tokenizer_llama = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_llama = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"meta-llama load took {time.time()-t0:.2f}s")
print("Llama 3.2 1B Instruct loaded")

meta-llama load took 144.33s
Llama 3.2 1B Instruct loaded


In [9]:
!nvidia-smi

Thu Sep 17 16:44:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             32W /   70W |    3273MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
#mem_after_a = get_gpu_mem_mb()
#model_a_usage = mem_after_a - base_mem
#print(f"Model A Memory: {model_a_usage:.2f} MB")

In [11]:
# llama verification
print("Running quick verification...")
_prompt = "Hi! How are you today?" # define a simple input string
# convert the prompt text to tokens, make it a pytorch tensor,
# and send it to the same device the model is on (the GPU)
_inputs = tokenizer_llama(_prompt, return_tensors="pt").to(model_llama.device)

# ask the model to generate a response based on the input
# max_new_tokens limits the output length to 5 tokens for quick check
_outputs = model_llama.generate(**_inputs, max_new_tokens=5)

# decode the numeric output tokens back into readable text
# outputs[0] ges the first sequence in the batch
print("Llama Verification Ok:", tokenizer_llama.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Running quick verification...
Llama Verification Ok: <|begin_of_text|>Hi! How are you today? I'm excited to share


In [12]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 16:45:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             32W /   70W |    3323MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
#Now load guff and quantized version
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/Llama-3.2-1B-Instruct-Q8_0.gguf"

t0 = time.time()
llmlamaGuff = Llama(model_path=read_path, n_gpu_layers=-1, verbose=False)
print(f"GGUF load took {time.time()-t0:.2f}s")

GGUF load took 64.47s


In [14]:
t0 = time.time()
output = llmlamaGuff.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)
print(f"Inference took {time.time()-t0:.2f}s")
print(output["choices"][0]["message"]["content"])

Inference took 0.16s
Hello!


In [15]:
#mem_after_b = get_gpu_mem_mb()
#model_b_usage = mem_after_b - mem_after_a
#print(f"Model B Memory: {model_b_usage:.2f} MB")

In [16]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 16:46:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             33W /   70W |    4865MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
# --- Load Gemma 2 2B ---
local_model_path = "/home/jovyan/shared/models/google--gemma-2-2b-it"

tokenizer_gemma = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_gemma = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Gemmaload took {time.time()-t0:.2f}s")
print(f"Gemma 2 2B loaded")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gemmaload took 310.00s
Gemma 2 2B loaded


In [18]:
# gemini verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_gemma(_prompt, return_tensors="pt").to(model_gemma.device)
_outputs = model_gemma.generate(**_inputs, max_new_tokens=5)

print("Gemma Verification Ok:", tokenizer_gemma.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Running quick verification...
Gemma Verification Ok: <bos>Hi! How are you today? 😊

I'm


In [19]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 16:52:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |   10047MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
# =====  Now trty gguf
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/gemma-2-2b-it.q4_k_m.gguf"

t0 = time.time()
llmGemG = Llama(model_path=read_path, n_gpu_layers=-1, verbose =False)
print(f"GGUF load took {time.time()-t0:.2f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


GGUF load took 100.53s


In [21]:
t0 = time.time()
output = llmGemG.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)
print(f"Inference took {time.time()-t0:.2f}s")
print(output["choices"][0]["message"]["content"])

Inference took 0.13s
Hello! 



In [22]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 16:55:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |   12239MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
# --- Load Qwen 2.5 0.5B ---
local_model_path = "/home/jovyan/shared/models/Qwen--Qwen2.5-0.5B-Instruct"
t0 = time.time()
tokenizer_qwen = AutoTokenizer.from_pretrained(local_model_path)
model_qwen = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Qwenload took {time.time()-t0:.2f}s")
print("Qwen 2.5 0.5B loaded")

# qwen verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_qwen(_prompt, return_tensors="pt").to(model_qwen.device)
_outputs = model_qwen.generate(**_inputs, max_new_tokens=5)
print("Qwen Verification Ok:", tokenizer_qwen.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Qwenload took 50.19s
Qwen 2.5 0.5B loaded
Running quick verification...
Qwen Verification Ok: Hi! How are you today? I hope you’re having


In [24]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 16:56:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |   13233MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# ==== Now tryguff and quantized version
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/qwen2.5-0.5b-instruct-q8_0.gguf"

t0 = time.time()
llmQuenG = Llama(model_path=read_path, n_gpu_layers=-1, verbose=False)
print(f"GGUF load took {time.time()-t0:.2f}s")

GGUF load took 20.64s


In [25]:
t0 = time.time()
output = llmQuenG.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)
print(f"Inference took {time.time()-t0:.2f}s")
print(output["choices"][0]["message"]["content"])

Inference took 0.11s
Hello! It's a pleasure to meet you.


In [26]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Sep 17 16:56:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |   13247MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [27]:
import time # Import the time library

# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [28]:
# Define a function to handle text generation and timing
def generate_text_gguf(model, prompt, max_new_tokens):

    start_time = time.time() # Record the time before generation starts

    
# Create the messages list (OpenAI-compatible format)
    messages = [
           {"role": "user", "content": prompt}
    ]

# Generate a response using create_chat_completion
    response = model.create_chat_completion(
       messages = messages,
       max_tokens = max_new_tokens
    )

# Extract and print the response
    print("Response:")
    print(response["choices"][0]["message"]["content"])
    # Generate text using the model
    output =  response["choices"][0]["message"]["content"]
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    completion_tokens = response["usage"]["completion_tokens"]
    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = len(output) / completion_tokens
    print(f"Generated {completion_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens per sec)")

   
    # Return the text and speed for potential later use
    return output, tokens_per_sec

In [29]:
prompt = "Describe the year-round climate in Paris, France"

In [30]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated 414 tokens in 10.96 seconds (37.77 tokens/sec)
Output:
.

Paris, the capital of France, is known for its mild and temperate climate, making it a desirable place to live and visit year-round. The city's climate is characterized by:

1. Mild winters: Paris has a mild winter, with temperatures ranging from 32°F (0°C) to 45°F (7°C). The average temperature in January, the coldest month, is around 38°F (3°C).
2. Mild springs: Spring in Paris is mild and pleasant, with temperatures gradually increasing from 45°F (7°C) to 60°F (16°C). The average temperature in March, the third month, is around 55°F (13°C).
3. Warm summers: Paris has a relatively mild summer, with temperatures ranging from 64°F (18°C) to 75°F (24°C). The average temperature in July, the hottest month, is around 73°F (23°C).
4. Cool autumns: Autumn in Paris is characterized by mild temperatures, with temperatures ranging from 50°F (10°C) to 65°F (18°C). The average temperature in September, the last month, is around 

In [31]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text_gguf(llmlamaGuff, prompt, max_new_tokens=500)

Response:
Paris, the capital of France, has a temperate oceanic climate, characterized by mild winters and cool summers. The city's climate is influenced by its location on the Seine River and the surrounding mountains, which create a microclimate with distinct seasonal variations.

**Seasonal Climate Patterns:**

1. **Winter (December to February):** Paris experiences a moderate climate during winter, with average temperatures ranging from 2°C (36°F) to 6°C (43°F) throughout the month. The city is known for its cold and snowy winters, with an average of 12 rainy days per month.
2. **Spring (March to May):** Spring in Paris is mild and pleasant, with temperatures gradually warming up to 15°C (59°F) to 20°C (68°F) during the month. The city experiences a moderate amount of rainfall, with an average of 12 rainy days per month.
3. **Summer (June to August):** Summer in Paris is warm and sunny, with average temperatures ranging from 18°C (64°F) to 23°C (73°F). The city experiences a modera

In [32]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500)

Generated 467 tokens in 31.18 seconds (14.98 tokens/sec)
Output:
.

Paris, France, enjoys a **temperate maritime climate**, which means it experiences mild, wet winters and warm, dry summers. 

Here's a breakdown:

**Winter (November - March):**
* **Temperature:** Average highs around 5°C (41°F), with lows around 0°C (32°F).
* **Precipitation:**  Significant rainfall, with an average of 100-150 mm (4-6 inches) per month.
* **Sunshine:**  Limited sunshine, with an average of 3-4 hours per day.

**Spring (April - May):**
* **Temperature:**  Gradually warming up, with average highs around 10°C (50°F).
* **Precipitation:**  Rainfall decreases, with an average of 50-70 mm (2-3 inches) per month.
* **Sunshine:**  Increasing sunshine, with an average of 6-8 hours per day.

**Summer (June - August):**
* **Temperature:**  Warm and sunny, with average highs around 25°C (77°F).
* **Precipitation:**  Very little rainfall, with an average of 20-30 mm (1-1.5 inches) per month.
* **Sunshine:**  Abund

In [33]:
# Generate with Gemma 2 2B GGUF
gemma_output, gemma_speed = generate_text_gguf(llmGemG, prompt, max_new_tokens=500)

Response:
Paris, France experiences a **temperate oceanic climate** characterized by mild, wet winters and warm, humid summers. 

Here's a breakdown:

**Temperature:**

* **Winter (December - February):** Average temperatures range from 3°C (37°F) to 7°C (45°F). Snowfall is common, with an average of around 10 inches. 
* **Spring (March - May):** Temperatures gradually rise, reaching highs of 15°C (59°F) and lows of 5°C (41°F). 
* **Summer (June - August):** Warm and humid, with average highs of 25°C (77°F) and lows of 15°C (59°F). 
* **Autumn (September - November):** Temperatures cool down, with average highs of 15°C (59°F) and lows of 8°C (46°F). 

**Rainfall:**

* **Year-round:** Paris receives an average of 600mm (23.6 inches) of rainfall annually. 
* **Winter:** The wettest months are December and January, with an average of 100mm (3.9 inches) of rainfall. 
* **Spring and Summer:** Rainfall is less frequent and generally lighter. 

**Sunshine:**

* **Year-round:** Paris receives 

In [34]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt, max_new_tokens=500)

Generated 500 tokens in 19.57 seconds (25.55 tokens/sec)
Output:
. The year-round climate in Paris is warm and humid with average temperatures ranging from 15°C to 20°C (59°F to 68°F) during the day and dropping to around 5°C to 7°C (41°F to 45°F) at night. Spring and summer are typically hot and dry, while fall and winter are mild and wet.

In spring, temperatures can reach up to 25°C (77°F), while in summer, they drop to around 15°C (59°F). Fall brings cooler temperatures, often falling between 10°C (50°F) and 15°C (59°F), with some days reaching below freezing. Winter sees temperatures ranging from -3°C (-27°F) in January to 2°C (-4°F) in July, with occasional snowfall.

The city's geography influences its climate, with many of its landmarks facing north and experiencing more direct sunlight throughout the year. This allows for a consistent pattern of temperature fluctuations, which contributes to the overall pleasant climate in Paris. However, it also means that Paris has a relativ

In [35]:
# Generate with Qwen 2.5 0.5B GGUF
qwen_output, qwen_speed = generate_text_gguf(llmQuenG, prompt, max_new_tokens=500)

Response:
Paris, France, is a city known for its diverse climate, which can vary significantly from season to season. Here's a general overview of the year-round climate in Paris:

1. Spring (March to May): This is the time when the weather becomes more pleasant and the days are longer. The temperature ranges from 15°C to 25°C, with a high of 28°C in the morning and a low of 12°C in the evening. The weather is generally sunny and clear, with occasional showers.

2. Summer (June to August): This is the peak season for outdoor activities, with temperatures ranging from 25°C to 30°C. The weather is generally warm and comfortable, with occasional showers and a high of 30°C in the morning and a low of 15°C in the evening.

3. Autumn (September to November): This is the time when the weather becomes cooler and the days are shorter. The temperature ranges from 10°C to 15°C, with a high of 12°C in the morning and a low of 8°C in the evening. The weather is generally sunny and clear, with occas

In [36]:
prompt_2 = """Scenario:
Four colored cups – Red, Blue, Green, and Yellow – are arranged in a circle.
One cup has a hidden star.

Facts:

The cup with the star is not Red.
The cup with the star is directly next to the Blue cup.
The Green cup is directly between the Yellow cup and the Red cup (when going around the circle in one direction).
Question:
Which color cup has the star? Just give the correct answer"""

In [37]:
prompt_2b = """Scenario:
There are three blocks. 
Block 1 is directly on the table.
Block 2 is directly on block 1.
Block 3 is directly  on block 2.

Facts:
Block 1 is blue.
Block  3 is green.
The middle block can be any color.


Question:
Is there a green block directly ontop of a non-green block? Justify the answer."""

In [38]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated 310 tokens in 8.20 seconds (37.82 tokens/sec)
Output:
.

## Step 1: Analyzing the given information
From the information given, we know the cup with the star is not Red, is directly next to the Blue cup, and is between the Yellow cup and the Red cup when moving around the circle in one direction.

## Step 2: Identifying possible positions for the star
Since the cup with the star is directly next to the Blue cup and is between the Yellow cup and the Red cup, and considering the cups are arranged in a circle, the only possible position for the star is next to the Blue cup and also next to the Red cup.

## Step 3: Eliminating impossible positions
However, if the star were directly next to the Red cup, it would also be next to the Yellow cup, which contradicts the statement that the cup with the star is directly next to the Blue cup. Therefore, the star cannot be next to the Red cup.

## Step 4: Considering the remaining options
Given that the star cannot be next to the Red cup, 

In [41]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2b, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated 1 tokens in 0.20 seconds (5.10 tokens/sec)
Output:



In [40]:
# Generate with Llama 3.2 1B GGuf
llama_output, llama_speed = generate_text_gguf(llmlamaGuff, prompt_2b, max_new_tokens=500)

Response:
To answer this question, let's analyze the given information:

1. Block 1 is blue.
2. Block 3 is green.
3. The middle block can be any color.

From the given information, we know that Block 1 is blue and Block 3 is green. Since Block 3 is green, it is directly on top of Block 1.

Now, let's consider the middle block. We don't know its color, but we do know that it can be any color. 

Since Block 1 is blue and Block 3 is green, and Block 3 is directly on top of Block 1, we can conclude that Block 1 is directly on top of the middle block.

Therefore, the answer to the question is: Yes, there is a green block directly on top of a non-green block.
Generated 170 tokens in 1.49 seconds (3.83 tokens per sec)


In [42]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2, max_new_tokens=500)

Generated 176 tokens in 12.01 seconds (14.65 tokens/sec)
Output:
.

Yellow


Blue


Green


Red


Answer:
**Yellow** 


Explanation:
Here's how we can solve this logic puzzle:

1. **The cup with the star is not Red.**  This eliminates one possibility.

2. **The cup with the star is directly next to the Blue cup.** This gives us a starting point.

3. **The Green cup is directly between the Yellow cup and the Red cup.** This helps us visualize the arrangement.

4. **Combining the information:**  We know the star is not Red, and it's next to Blue.  The Green cup is in the middle, and the Yellow cup and Red cup are on either side.  This means the star must be on the Yellow cup. 


Let me know if you'd like to try another puzzle! 



In [43]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2b, max_new_tokens=500)

Generated 118 tokens in 8.09 seconds (14.59 tokens/sec)
Output:


**Solution:**

Yes, there is a green block directly on top of a non-green block. Here's why:

* **Block 3 is green:** This is given.
* **Block 3 is directly on top of Block 2:** This is also given.
* **Block 2 is directly on top of Block 1:** This is given.

Therefore, Block 3 is directly on top of a non-green block (Block 2). 


Let me know if you'd like to explore other scenarios! 



In [44]:
# Generate with Gemma 2 2B GGuf
gemma_output, gemma_speed = generate_text_gguf(llmGemG, prompt_2, max_new_tokens=500)

Response:
Blue 

Generated 3 tokens in 0.14 seconds (2.00 tokens per sec)


In [45]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_2, max_new_tokens=500)

Generated 500 tokens in 19.68 seconds (25.40 tokens/sec)
Output:
. If there are multiple answers, choose all of them separated by commas.
Red
Blue
Green
Yellow

To solve this problem, we need to analyze the given facts step-by-step to determine which color cup has the hidden star.

1. **Fact 1:** The cup with the star is not Red.
   - This means that if the star were in any other color, it would be Red.

2. **Fact 2:** The cup with the star is directly next to the Blue cup.
   - Since the star cannot be in Red, it must be either Blue or Green.

3. **Fact 3:** The Green cup is directly between the Yellow cup and the Red cup (when going around the circle in one direction).
   - This implies that the Green cup is also directly next to the Yellow cup, making them adjacent on the circle.

Given these facts, let's consider each possibility for the position of the star:

- **Case 1: Star is in Red**
  - This satisfies Fact 1 because Red cannot have the star.
  - Now, we need to check if the B

In [46]:
# Generate with Qwen 2.5 0.5B GGUF
qwen_output, qwen_speed = generate_text_gguf(llmQuenG, prompt_2, max_new_tokens=500)

Response:
Yellow
Generated 1 tokens in 0.04 seconds (6.00 tokens per sec)


In [ ]:
prompt_3 = """Write a simple Python function called `calculate_area`
that accepts the length and width of a rectangle and returns the area."""

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Llama 3.2 1B GGuf
llama_output, llama_speed = generate_text_gguf(llmlamaGuff, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B GGuf
gemma_output, gemma_speed = generate_text_gguf(llmGemG, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B gguf
qwen_output, qwen_speed = generate_text_gguf(llmQuenG, prompt_3, max_new_tokens=500)